In [0]:
%sql
-- =============================================================
--  GEO SAFRAS · Schema v1.0 · Junho 2026
--  Databricks SQL — catalog: workspace
--  Schemas: gs_bronze · gs_silver · gs_gold
--  NOVIDADE: tabela usuario + isolamento por usuario_id
-- =============================================================

-- ─────────────────────────────────────────────────────────────
--  BRONZE  (strings brutas, sem DEFAULT — evita
--           DELTA_MULTIPLE_SOURCE_ROWS e feature errors)
-- ─────────────────────────────────────────────────────────────

-- 1. USUÁRIO
CREATE TABLE IF NOT EXISTS workspace.gs_bronze.usuario (
  id               STRING,
  nome_completo    STRING,
  usuario          STRING,
  senha_hash       STRING,
  estado           STRING,
  cidade           STRING,
  email            STRING,
  ativo            STRING,
  primeiro_acesso  STRING,
  created_at       STRING,
  updated_at       STRING,
  source_file      STRING,
  ingestion_timestamp STRING
);

-- 2. FAZENDA  (agora com usuario_id)
CREATE TABLE IF NOT EXISTS workspace.gs_bronze.fazenda (
  id               STRING,
  usuario_id       STRING,
  nome             STRING,
  municipio        STRING,
  estado           STRING,
  created_at       STRING,
  updated_at       STRING,
  source_file      STRING,
  ingestion_timestamp STRING
);

-- 3. TALHÃO  (sem alteração de colunas, mas isolado via fazenda→usuario)
CREATE TABLE IF NOT EXISTS workspace.gs_bronze.talhao (
  id               STRING,
  fazenda_id       STRING,
  nome             STRING,
  area_ha          STRING,
  geom_wkt         STRING,
  created_at       STRING,
  updated_at       STRING,
  source_file      STRING,
  ingestion_timestamp STRING
);

-- 4. ANÁLISE DE SOLO — adiciona usuario_id e agronomo_outro
CREATE TABLE IF NOT EXISTS workspace.gs_bronze.analise_solo (
  id               STRING,
  usuario_id       STRING,
  talhao_id        STRING,
  data_coleta      STRING,
  laboratorio      STRING,
  extrator_p       STRING,
  agronomo         STRING,
  agronomo_outro   STRING,
  observacoes      STRING,
  status           STRING,
  app_version      STRING,
  created_at       STRING,
  updated_at       STRING,
  source_file      STRING,
  ingestion_timestamp STRING
);

-- 5. CAMADA — adiciona usuario_id
CREATE TABLE IF NOT EXISTS workspace.gs_bronze.analise_solo_camada (
  id               STRING,
  analise_id       STRING,
  usuario_id       STRING,
  profundidade     STRING,
  ph_cacl2         STRING,
  ph_h2o           STRING,
  h_al             STRING,
  al_toxico        STRING,
  v_pct            STRING,
  m_pct            STRING,
  n_mg             STRING,
  p_mg             STRING,
  k_mg             STRING,
  ca_cmol          STRING,
  mg_cmol          STRING,
  s_mg             STRING,
  sb_cmol          STRING,
  b_mg             STRING,
  zn_mg            STRING,
  cu_mg            STRING,
  fe_mg            STRING,
  mn_mg            STRING,
  mo_g             STRING,
  ctc_total        STRING,
  ctc_efetiva      STRING,
  argila_pct       STRING,
  areia_pct        STRING,
  silte_pct        STRING,
  created_at       STRING,
  source_file      STRING,
  ingestion_timestamp STRING
);

-- 6. AJUSTE DE DOSE — adiciona usuario_id
CREATE TABLE IF NOT EXISTS workspace.gs_bronze.ajuste_dose (
  id               STRING,
  usuario_id       STRING,
  analise_id       STRING,
  talhao_id        STRING,
  safra            STRING,
  cultura          STRING,
  pct_sementes     STRING,
  pct_n            STRING,
  pct_p2o5         STRING,
  pct_k2o          STRING,
  pct_calcario     STRING,
  pct_gesso        STRING,
  pct_enxofre      STRING,
  dose_sementes_kg STRING,
  dose_n_kg        STRING,
  dose_p2o5_kg     STRING,
  dose_k2o_kg      STRING,
  dose_calcario_tha STRING,
  dose_gesso_tha   STRING,
  dose_enxofre_kg  STRING,
  agronomo         STRING,
  observacoes      STRING,
  created_at       STRING,
  updated_at       STRING,
  source_file      STRING,
  ingestion_timestamp STRING
);


-- ─────────────────────────────────────────────────────────────
--  SILVER  (tipos corretos · deduplicação · isolamento)
-- ─────────────────────────────────────────────────────────────

CREATE TABLE IF NOT EXISTS workspace.gs_silver.usuario (
  id               BIGINT,
  nome_completo    STRING,
  usuario          STRING,
  senha_hash       STRING,
  estado           STRING,
  cidade           STRING,
  email            STRING,
  ativo            BOOLEAN,
  primeiro_acesso  BOOLEAN,
  created_at       TIMESTAMP,
  updated_at       TIMESTAMP
);

CREATE TABLE IF NOT EXISTS workspace.gs_silver.fazenda (
  id               BIGINT,
  usuario_id       BIGINT,
  nome             STRING,
  municipio        STRING,
  estado           STRING,
  created_at       TIMESTAMP,
  updated_at       TIMESTAMP
);

CREATE TABLE IF NOT EXISTS workspace.gs_silver.talhao (
  id               BIGINT,
  fazenda_id       BIGINT,
  nome             STRING,
  area_ha          DOUBLE,
  geom_wkt         STRING,
  created_at       TIMESTAMP,
  updated_at       TIMESTAMP
);

CREATE TABLE IF NOT EXISTS workspace.gs_silver.analise_solo (
  id               BIGINT,
  usuario_id       BIGINT,
  talhao_id        BIGINT,
  data_coleta      DATE,
  laboratorio      STRING,
  extrator_p       STRING,
  agronomo         STRING,
  agronomo_outro   STRING,
  observacoes      STRING,
  status           STRING,
  app_version      STRING,
  created_at       TIMESTAMP,
  updated_at       TIMESTAMP
);

CREATE TABLE IF NOT EXISTS workspace.gs_silver.analise_solo_camada (
  id               BIGINT,
  analise_id       BIGINT,
  usuario_id       BIGINT,
  profundidade     STRING,
  ph_cacl2         DOUBLE,
  ph_h2o           DOUBLE,
  h_al             DOUBLE,
  al_toxico        DOUBLE,
  v_pct            DOUBLE,
  m_pct            DOUBLE,
  n_mg             DOUBLE,
  p_mg             DOUBLE,
  k_mg             DOUBLE,
  ca_cmol          DOUBLE,
  mg_cmol          DOUBLE,
  s_mg             DOUBLE,
  sb_cmol          DOUBLE,
  b_mg             DOUBLE,
  zn_mg            DOUBLE,
  cu_mg            DOUBLE,
  fe_mg            DOUBLE,
  mn_mg            DOUBLE,
  mo_g             DOUBLE,
  ctc_total        DOUBLE,
  ctc_efetiva      DOUBLE,
  argila_pct       DOUBLE,
  areia_pct        DOUBLE,
  silte_pct        DOUBLE,
  created_at       TIMESTAMP
);

CREATE TABLE IF NOT EXISTS workspace.gs_silver.ajuste_dose (
  id               BIGINT,
  usuario_id       BIGINT,
  analise_id       BIGINT,
  talhao_id        BIGINT,
  safra            STRING,
  cultura          STRING,
  pct_sementes     DOUBLE,
  pct_n            DOUBLE,
  pct_p2o5         DOUBLE,
  pct_k2o          DOUBLE,
  pct_calcario     DOUBLE,
  pct_gesso        DOUBLE,
  pct_enxofre      DOUBLE,
  dose_sementes_kg DOUBLE,
  dose_n_kg        DOUBLE,
  dose_p2o5_kg     DOUBLE,
  dose_k2o_kg      DOUBLE,
  dose_calcario_tha DOUBLE,
  dose_gesso_tha   DOUBLE,
  dose_enxofre_kg  DOUBLE,
  agronomo         STRING,
  observacoes      STRING,
  created_at       TIMESTAMP,
  updated_at       TIMESTAMP
);


-- ─────────────────────────────────────────────────────────────
--  GOLD  (views para Tableau · já filtráveis por usuario_id)
-- ─────────────────────────────────────────────────────────────

CREATE OR REPLACE VIEW workspace.gs_gold.vw_analise_tableau AS
SELECT
  u.id                          AS usuario_id,
  u.nome_completo               AS usuario_nome,
  f.id                          AS fazenda_id,
  f.nome                        AS fazenda,
  f.municipio,
  f.estado,
  t.id                          AS talhao_id,
  t.nome                        AS talhao,
  t.area_ha,
  a.id                          AS analise_id,
  a.data_coleta,
  a.laboratorio,
  a.extrator_p,
  a.agronomo,
  a.agronomo_outro,
  a.status,
  a.observacoes,
  c.profundidade,
  c.ph_cacl2, c.ph_h2o, c.h_al, c.al_toxico, c.v_pct, c.m_pct,
  c.n_mg, c.p_mg, c.k_mg,
  c.ca_cmol, c.mg_cmol, c.s_mg, c.sb_cmol,
  c.b_mg, c.zn_mg, c.cu_mg, c.fe_mg, c.mn_mg,
  c.mo_g, c.ctc_total, c.ctc_efetiva,
  c.argila_pct, c.areia_pct, c.silte_pct
FROM workspace.gs_silver.analise_solo a
JOIN workspace.gs_silver.talhao              t ON t.id = a.talhao_id
JOIN workspace.gs_silver.fazenda             f ON f.id = t.fazenda_id
JOIN workspace.gs_silver.usuario             u ON u.id = a.usuario_id
JOIN workspace.gs_silver.analise_solo_camada c ON c.analise_id = a.id
WHERE a.status IN ('enviado','processado');


CREATE OR REPLACE VIEW workspace.gs_gold.vw_ajuste_doses_tableau AS
SELECT
  u.id                          AS usuario_id,
  u.nome_completo               AS usuario_nome,
  f.nome                        AS fazenda,
  t.nome                        AS talhao,
  t.area_ha,
  a.data_coleta,
  a.extrator_p,
  d.safra,
  d.cultura,
  d.pct_sementes, d.pct_n, d.pct_p2o5, d.pct_k2o,
  d.pct_calcario, d.pct_gesso, d.pct_enxofre,
  d.dose_sementes_kg, d.dose_n_kg, d.dose_p2o5_kg, d.dose_k2o_kg,
  d.dose_calcario_tha, d.dose_gesso_tha, d.dose_enxofre_kg,
  (d.dose_sementes_kg  * t.area_ha) AS total_sementes_kg,
  (d.dose_n_kg         * t.area_ha) AS total_n_kg,
  (d.dose_p2o5_kg      * t.area_ha) AS total_p2o5_kg,
  (d.dose_k2o_kg       * t.area_ha) AS total_k2o_kg,
  (d.dose_calcario_tha * t.area_ha) AS total_calcario_t,
  (d.dose_gesso_tha    * t.area_ha) AS total_gesso_t,
  (d.dose_enxofre_kg   * t.area_ha) AS total_enxofre_kg,
  d.agronomo,
  d.created_at                  AS data_ajuste
FROM workspace.gs_silver.ajuste_dose  d
JOIN workspace.gs_silver.analise_solo a ON a.id = d.analise_id
JOIN workspace.gs_silver.talhao       t ON t.id = d.talhao_id
JOIN workspace.gs_silver.fazenda      f ON f.id = t.fazenda_id
JOIN workspace.gs_silver.usuario      u ON u.id = d.usuario_id;

In [0]:
%sql
ALTER TABLE workspace.gs_bronze.usuario ADD COLUMN primeiro_acesso STRING;
ALTER TABLE workspace.gs_silver.usuario ADD COLUMN primeiro_acesso BOOLEAN;
UPDATE workspace.gs_bronze.usuario SET primeiro_acesso = 'true' WHERE primeiro_acesso IS NULL;

In [0]:
%sql
ALTER TABLE workspace.gs_bronze.usuario ADD COLUMN email STRING;
ALTER TABLE workspace.gs_silver.usuario ADD COLUMN email STRING;

In [0]:
%sql
select
    nome_completo,
    usuario,
    email
 from workspace.gs_bronze.usuario

In [0]:
%sql
UPDATE workspace.gs_silver.usuario SET email = 'cakanilo@gmail.com' WHERE usuario = 'danilo.silva';
UPDATE workspace.gs_silver.usuario SET email = 'patriciaedir@gmail.com' WHERE usuario = 'patricia.edir';

In [0]:
%sql
-- Limpar e regravar o email dos dois usuários
UPDATE workspace.gs_bronze.usuario 
SET email = 'cakanilo@gmail.com' 
WHERE usuario = 'danilo.silva';

UPDATE workspace.gs_bronze.usuario 
SET email = 'patriciaedir@gmail.com' 
WHERE usuario = 'patricia.edir';

In [0]:
%sql
UPDATE workspace.gs_bronze.usuario 
SET primeiro_acesso = 'true'
WHERE usuario = 'danilo.silva';

## criando as tabelas Pedidos, Saude_financeira e Calendario_safras

In [0]:
%sql
-- criando as tabelas Pedidos

-- ─── 1. PEDIDOS ───────────────────────────────────────────────
CREATE TABLE IF NOT EXISTS workspace.gs_bronze.pedidos (
 
  -- Identificação
  id_pedido               STRING,       -- 'BR-2025-133872'
  data_pedido             DATE,
  data_envio              DATE,
 
  -- Localização
  cultura                 STRING,       -- 'Milho'
  cidade                  STRING,       -- 'padre bernardo'
  localidade              STRING,       -- 'fazenda armazens gerais santa lucia'
  estado                  STRING,       -- 'go'
  pais                    STRING,       -- 'Brasil'
  regiao                  STRING,       -- 'Sul'
 
  -- Produto
  subcultura              STRING,       -- 'milho 1'
  subcultura2             STRING,       -- 'milho 1'
  formato                 STRING,       -- 'Quadrado/Retangular'
 
  -- Financeiro bruto
  vendas                  DOUBLE,       -- Receita total R$
  quantidade              DOUBLE,       -- Sacas (60kg)
  desconto                DOUBLE,       -- 0 a 1 (ex: 0.10 = 10%)
  lucro                   DOUBLE,
  custo_total             DOUBLE,
  margem_liquida_pct      DOUBLE,       -- %
  preco_por_unidade       DOUBLE,       -- R$/saca
  custo_por_unidade       DOUBLE,
  lucro_por_unidade       DOUBLE,
 
  -- Dimensões temporais
  ano                     INT,
  mes                     INT,
  trimestre               INT,
  safra                   STRING,       -- 'safrinha_2025', 'safra_verao_2025/2026'
 
  -- Categorias calculadas
  categoria_margem        STRING,       -- 'Boa', 'Moderada'
  impacto_desconto        DOUBLE,
 
  -- Custos estimados por categoria
  custo_insumos_est       DOUBLE,
  custo_fertilizantes_est DOUBLE,
  custo_defensivos_est    DOUBLE,
  custo_outros_est        DOUBLE,
 
  -- Comparativo CEPEA/ESALQ
  cepea_referencia_sc     DOUBLE,       -- Preço CEPEA R$/saca
  gap_preco_cepea         DOUBLE,       -- Diferença R$/saca vs CEPEA
  perc_vs_cepea           DOUBLE,       -- % acima/abaixo CEPEA
  receita_perdida_cepea   DOUBLE,
  lucro_justo_cepea       DOUBLE,
 
  -- Metadados de ingestão
  ingestion_timestamp     TIMESTAMP,
  source_file             STRING
)
USING DELTA
COMMENT 'Bronze: pedidos/vendas brutos da Fazenda Armazéns Gerais Santa Lúcia';

In [0]:
%sql

-- criando as tabelas Calendário

-- ─── 2. CALENDARIO_SAFRAS ─────────────────────────────────────
CREATE TABLE IF NOT EXISTS workspace.gs_bronze.calendario_safras (
 
  cultura                 STRING,       -- 'Milho'
  talhao                  STRING,       -- 'Talhão 510', 'Talhão sede'
  ano                     INT,
  safra_tipo              STRING,       -- 'safrinha', 'safra_verao'
  safra_ano_label         STRING,       -- '2025', '2025/2026'
 
  -- Mês
  mes_num                 INT,          -- 1-12
  mes_nome                STRING,       -- 'Jan', 'Fev', ...
  mes_ordem_exibicao      INT,          -- Ordem relativa no calendário agrícola
  mes_data_inicio         DATE,
  mes_data_fim            DATE,
 
  -- Estágio fenológico
  estagio                 STRING,       -- 'Plantio', 'Desenvolvimento', 'Floração', 'Enchimento', 'Colheita'
  ordem_estagio           INT,          -- 1=Plantio ... 5=Colheita
  cor_hex                 STRING,       -- Cor para visualização Tableau
  data_inicio_estagio     DATE,
  data_fim_estagio        DATE,
 
  -- Metadados
  ingestion_timestamp     TIMESTAMP,
  source_file             STRING
)
USING DELTA
COMMENT 'Bronze: calendário fenológico das safras por talhão';


In [0]:
%sql
-- Criando a tabela Saude financeira

-- ─── 3. SAUDE_FINANCEIRA ──────────────────────────────────────
CREATE TABLE IF NOT EXISTS workspace.gs_bronze.saude_financeira (
 
  fazenda                             STRING,
  ano                                 INT,
 
  -- P&L
  receita_total                       DOUBLE,
  custo_total                         DOUBLE,
  lucro_total                         DOUBLE,
 
  -- Balanço patrimonial (base)
  passivo_circulante_base             DOUBLE,
  ativo_circulante_base               DOUBLE,
  capital_giro_base                   DOUBLE,
  liquidez_corrente_base              DOUBLE,
  area_ha                             INT,
 
  -- Simulação patrimonial
  ativo_total_simulado                DOUBLE,
  passivo_nao_circulante_simulado     DOUBLE,
  endividamento_geral_pct             DOUBLE,
 
  -- Simulação financeira
  despesas_financeiras_simuladas      DOUBLE,
  cobertura_juros_x                   DOUBLE,
 
  -- Score de risco
  score_risco_financeiro              INT,          -- 0-100
  score_risco_classificacao           STRING,       -- 'Baixo', 'Moderado', 'Alto'
 
  -- Premissa do cenário
  premissa                            STRING,       -- 'CENARIO SIMULADO - ...'
 
  -- Metadados
  ingestion_timestamp                 TIMESTAMP,
  source_file                         STRING
)
USING DELTA
COMMENT 'Bronze: saúde financeira simulada por fazenda/ano — CENÁRIO SIMULADO';